<a href="https://colab.research.google.com/github/Sosswood/Machine-Learning/blob/main/Unit_08_Recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#installing the required libraries. If not using Google Colab(recommended), make sure to check the requirements & versions

!pip install pandas numpy scikit-learn shap tqdm


In [2]:
# Load, examine and prepare dataset

import pandas as pd
import numpy as np
import zipfile
import io
import requests
from pathlib import Path


# Download MovieLens 100K
URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = Path("ml-100k")

if not DATA_DIR.exists():
    print("Downloading MovieLens 100K...")
    r = requests.get(URL)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall()

print("Dataset ready.")

# If your network blocks downloads, requests.get() will fail.
# Check your connection and/or any other access issues.


# Load ratings
ratings = pd.read_csv(
    DATA_DIR / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)


# Load items
items = pd.read_csv(
    DATA_DIR / "u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

item_cols = (
    ["item_id", "title", "release_date", "video_release_date", "imdb_url"]
    + [f"genre_{i}" for i in range(19)]
)
items.columns = item_cols


# Validation & diagnostics
print(ratings.head())
print(items[["item_id", "title"]].head())

assert ratings["rating"].between(1, 5).all()
assert ratings["user_id"].nunique() <= 1000
assert ratings["item_id"].nunique() <= 1700

print("Ratings:", len(ratings))
print("Users:", ratings["user_id"].nunique())
print("Items:", ratings["item_id"].nunique())
print("Min/Max timestamp:", ratings["timestamp"].min(), ratings["timestamp"].max())

#Validation is essential. Recommenders are extremely sensitive to wrong/missing columns etc.
# REMEMBER: if your data is wrong, the model might still run, but your results will be meaningless

# Sparsity
# Sparsity measures how empty the user–movie matrix is, which can lead to less reliable recommendations.
# Most recommender datasets tend to be sparse, but this an important metrics to check.

n_users = ratings["user_id"].nunique()
n_items = ratings["item_id"].nunique()
sparsity = 1 - (len(ratings) / (n_users * n_items))
print(f"Sparsity: {sparsity:.4f}")

print("Rating distribution:")
print(ratings["rating"].value_counts().sort_index())


Dataset ready.
   user_id  item_id  rating  timestamp
0      196      242       3  881250949
1      186      302       3  891717742
2       22      377       1  878887116
3      244       51       2  880606923
4      166      346       1  886397596
   item_id              title
0        1   Toy Story (1995)
1        2   GoldenEye (1995)
2        3  Four Rooms (1995)
3        4  Get Shorty (1995)
4        5     Copycat (1995)
Ratings: 100000
Users: 943
Items: 1682
Min/Max timestamp: 874724710 893286638
Sparsity: 0.9370
Rating distribution:
rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [3]:
# Time-aware train/val/test split (80/10/10)
ratings_sorted = ratings.sort_values("timestamp").reset_index(drop=True)
n = len(ratings_sorted)

train = ratings_sorted.iloc[: int(0.8*n)]
val   = ratings_sorted.iloc[int(0.8*n): int(0.9*n)]
test  = ratings_sorted.iloc[int(0.9*n):]

assert train["timestamp"].max() <= val["timestamp"].min()
assert val["timestamp"].max() <= test["timestamp"].min()

print("\nSplit sizes:", len(train), len(val), len(test))

#Why time aware? In real life, you recommend based on the past, not the future.
# A random split can accidentally let the model “learn” patterns that rely on future interactions (data leakage).


Split sizes: 80000 10000 10000


In [4]:
# Build a Baseline Model.
# They are a benchmark against which more complex models can be evaluated.

from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np # Import numpy if not already imported, for np.sqrt

#Baselines (sanity checks)
def eval_constant(df, c):
    y_true = df["rating"].values.astype(float)
    y_pred = np.full_like(y_true, c, dtype=float)
    # Calculate MSE first, then take the square root for RMSE
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

global_mean = train["rating"].mean()
print("\nBaseline(global mean) VAL:", eval_constant(val, global_mean))
print("Baseline(global mean) TEST:", eval_constant(test, global_mean))

# Bias baseline: r_hat = mu + b_u + b_i (with regularization)
mu = train["rating"].mean()
lambda_reg = 10.0

user_sum = train.groupby("user_id")["rating"].sum()
user_cnt = train.groupby("user_id")["rating"].count()
item_sum = train.groupby("item_id")["rating"].sum()
item_cnt = train.groupby("item_id")["rating"].count()

b_u = ((user_sum - user_cnt * mu) / (user_cnt + lambda_reg)).to_dict()
b_i = ((item_sum - item_cnt * mu) / (item_cnt + lambda_reg)).to_dict()

def predict_bias(df):
    bu = df["user_id"].map(b_u).fillna(0.0).values
    bi = df["item_id"].map(b_i).fillna(0.0).values
    return mu + bu + bi

def eval_model(df, pred_fn):
    y_true = df["rating"].values.astype(float)
    y_pred = pred_fn(df).astype(float)
    # Calculate MSE first, then take the square root for RMSE
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

print("\nBias baseline VAL:", eval_model(val, predict_bias))
print("\nBias baseline TEST:", eval_model(test, predict_bias))

# Bias account for people's different approach.
# Expected outcome: bias baseline should beat global mean baseline.


Baseline(global mean) VAL: (np.float64(1.0979175162552057), 0.92612078)
Baseline(global mean) TEST: (np.float64(1.1399315560593977), 0.96937963)

Bias baseline VAL: (np.float64(1.0191052016857542), 0.810877103220018)

Bias baseline TEST: (np.float64(1.0446204580678322), 0.8480805896543465)


In [5]:
# Matrix Factorization assumes each user and each movie can be described by hidden features
#We don’t label these features. The model discovers them automatically.

import numpy as np
from tqdm import trange

# Define the mapping function to convert user/item IDs to contiguous indices
user_ids = sorted(ratings["user_id"].unique())
item_ids = sorted(ratings["item_id"].unique())

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_idx = {item_id: idx for idx, item_id in enumerate(item_ids)}

def to_idx(df):
    u_idx = df["user_id"].map(user_to_idx).values
    m_idx = df["item_id"].map(item_to_idx).values
    r = df["rating"].values
    return u_idx, m_idx, r

#Matrix Factorization (SGD==stochastic gradient descent)

train_u, train_m, train_r = to_idx(train)
val_u,   val_m,   val_r   = to_idx(val)
test_u,  test_m,  test_r  = to_idx(test)

n_users = len(user_ids)
n_items = len(item_ids)

rng = np.random.default_rng(42)
k = 32
lr = 0.01
reg = 0.02
epochs = 30

mu_mf = train_r.mean()
bu = np.zeros(n_users)
bi = np.zeros(n_items)
P = 0.1 * rng.standard_normal((n_users, k))
Q = 0.1 * rng.standard_normal((n_items, k))

def mf_predict(u_idx, m_idx):
    return mu_mf + bu[u_idx] + bi[m_idx] + np.sum(P[u_idx] * Q[m_idx], axis=1)

def rmse_vec(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

history = []
print("\nTraining MF...")
for epoch in trange(1, epochs + 1):
    perm = rng.permutation(len(train_r))
    u_sh, m_sh, r_sh = train_u[perm], train_m[perm], train_r[perm]

    for u, m, r in zip(u_sh, m_sh, r_sh):
        pred = mu_mf + bu[u] + bi[m] + P[u].dot(Q[m])
        err = r - pred

        bu[u] += lr * (err - reg * bu[u])
        bi[m] += lr * (err - reg * bi[m])

        pu = P[u].copy()
        qi = Q[m].copy()
        P[u] += lr * (err * qi - reg * pu)
        Q[m] += lr * (err * pu - reg * qi)

    train_rmse = rmse_vec(train_r, mf_predict(train_u, train_m))
    val_rmse   = rmse_vec(val_r,   mf_predict(val_u,   val_m))
    history.append((epoch, train_rmse, val_rmse))

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d} | train RMSE={train_rmse:.4f} | val RMSE={val_rmse:.4f}")

test_pred = mf_predict(test_u, test_m)
test_rmse = rmse_vec(test_r, test_pred)
test_mae  = float(np.mean(np.abs(test_r - test_pred)))
print("\nMF TEST RMSE:", test_rmse, " MAE:", test_mae)


Training MF...


  3%|▎         | 1/30 [00:01<00:49,  1.72s/it]

Epoch 01 | train RMSE=0.9554 | val RMSE=1.0388


 17%|█▋        | 5/30 [00:07<00:36,  1.47s/it]

Epoch 05 | train RMSE=0.8905 | val RMSE=1.0301


...

Epoch 20 | train RMSE=0.6338 | val RMSE=1.0381


 83%|████████▎ | 25/30 [00:41<00:08,  1.74s/it]

Epoch 25 | train RMSE=0.5813 | val RMSE=1.0404


100%|██████████| 30/30 [00:48<00:00,  1.63s/it]

Epoch 30 | train RMSE=0.5458 | val RMSE=1.0429

MF TEST RMSE: 1.0399901844123336  MAE: 0.8358307064794835


It is important here to look at your results.
Training diagnostics: train RMSE vs val RMSE
If both go down → good.
BUT
If train goes down but val goes up → overfitting.
Is it the case here? Check.
If you detect overfitting, one simple fix is to stop early (fewer epochs) in your training routine.

In [6]:
# Top-N recommendations + ranking validation
#Why do we use a ranking metric? Because of what the RS do.
# They are judged on their usefulness: “Did we put the right movies near the top?”


train_items_by_user = train.groupby("user_id")["item_id"].apply(set).to_dict()
test_pos = test[test["rating"] >= 4].groupby("user_id")["item_id"].apply(set).to_dict()

all_item_idx = np.arange(n_items)

u2i = user_to_idx # Alias user_to_idx to u2i
m2i = item_to_idx # Alias item_to_idx to m2i

def ndcg_at_k(relevances, k):
    rel = relevances[:k]
    dcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rel))
    ideal = sorted(relevances, reverse=True)[:k]
    idcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else float(dcg / idcg)

def recommend_for_user(user_id, K=10):
    uidx = u2i[user_id]
    seen = train_items_by_user.get(user_id, set())
    seen_idx = np.array([m2i[m] for m in seen if m in m2i], dtype=int)

    u_vec = np.full(n_items, uidx, dtype=int)
    scores = mf_predict(u_vec, all_item_idx)

    scores[seen_idx] = -1e9  # filter seen items

    topk_idx = np.argpartition(-scores, K)[:K]
    topk_idx = topk_idx[np.argsort(-scores[topk_idx])]

    topk_item_ids = [item_ids[i] for i in topk_idx]
    return topk_item_ids, scores[topk_idx]

def ranking_eval(K=10, max_users=500):
    users = list(test["user_id"].unique())[:max_users]
    precs, recs, ndcgs, cover_items = [], [], [], set()

    for u in users:
        gt = test_pos.get(u, set())
        if not gt:
            continue

        recs_u, _ = recommend_for_user(u, K=K)
        cover_items.update(recs_u)

        hits = [1 if m in gt else 0 for m in recs_u]
        precs.append(np.mean(hits))
        recs.append(np.sum(hits) / len(gt))
        ndcgs.append(ndcg_at_k(hits, K))

    coverage = len(cover_items) / n_items
    return {
        "precision@k": float(np.mean(precs)) if precs else 0.0,
        "recall@k": float(np.mean(recs)) if recs else 0.0,
        "ndcg@k": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "item_coverage": float(coverage),
        "users_evaluated": int(len(precs))
    }

print("\nRanking metrics:", ranking_eval(K=10))


Ranking metrics: {'precision@k': 0.14183006535947715, 'recall@k': 0.051162096261366884, 'ndcg@k': 0.30398477864123846, 'item_coverage': 0.14625445897740785, 'users_evaluated': 153}


Metrics explained:

Precision@K: among the top K recommendations, what fraction are relevant?

Recall@K: among the relevant items the user had, what fraction did we retrieve in top K?

NDCG@K: like precision but rewards correct items appearing earlier in the list (higher rank is better).

Item coverage: how many distinct items appear in recommendations across users.

Low coverage means the model recommends the same popular movies to everyone.

In [8]:
# Diagnostics: cold-start slices + popularity bias proxy
# It is important to classify the users in RS
# Users with few ratings are hard (COLD USERS): not enough data to learn a stable preference vector
# This is why we compute RMSE separately.

user_counts = train.groupby("user_id")["rating"].count()

def slice_users(min_ratings, max_ratings=None):
    if max_ratings is None:
        return set(user_counts[user_counts >= min_ratings].index)
    return set(user_counts[(user_counts >= min_ratings) & (user_counts <= max_ratings)].index)

cold_users = slice_users(1, 10)
warm_users = slice_users(50, None)

def rmse_on_user_subset(df, user_set):
    df2 = df[df["user_id"].isin(user_set)]
    if len(df2) == 0:
        return None
    u, m, r = to_idx(df2)
    pred = mf_predict(u, m)
    return rmse_vec(r, pred)

print("\nRMSE test cold users (1-10 ratings):", rmse_on_user_subset(test, cold_users))
print("RMSE test warm users (>=50 ratings):", rmse_on_user_subset(test, warm_users))

item_pop = train.groupby("item_id")["rating"].count().to_dict()

def avg_popularity_of_recs(sample_users=200, K=10):
    users = list(test["user_id"].unique())[:sample_users]
    pops = []
    for u in users:
        recs_u, _ = recommend_for_user(u, K=K)
        pops.extend([item_pop.get(m, 0) for m in recs_u])
    return float(np.mean(pops)), float(np.median(pops))

print("Avg/Median popularity of recs:", avg_popularity_of_recs())


RMSE test cold users (1-10 ratings): 1.2365472979247374
RMSE test warm users (>=50 ratings): 1.0095493780701932
Avg/Median popularity of recs: (159.83072289156627, 148.0)


In [9]:
from sklearn.linear_model import Ridge
import shap

# Explainable AI (SHAP) for MF prediction
def mf_feature_matrix(u_idx, m_idx):
    inter = P[u_idx] * Q[m_idx]  # (n, k)
    return np.column_stack([bu[u_idx], bi[m_idx], inter])

# Surrogate training sample (approximate MF behavior)
N_SAMPLE = 20000
idx = rng.choice(len(train_r), size=min(N_SAMPLE, len(train_r)), replace=False)

X_surr = mf_feature_matrix(train_u[idx], train_m[idx])
y_surr = mf_predict(train_u[idx], train_m[idx])

surr = Ridge(alpha=1.0)
surr.fit(X_surr, y_surr)

pred_check = surr.predict(X_surr)
print("\nSurrogate fidelity RMSE (surrogate vs MF):", rmse_vec(y_surr, pred_check))

# SHAP for linear surrogate
explainer = shap.LinearExplainer(surr, X_surr, feature_perturbation="interventional")
feature_names = ["user_bias", "item_bias"] + [f"latent_inter_{j}" for j in range(k)]

def explain_prediction(user_id, item_id, top_n=10):
    uidx = u2i[user_id]
    midx = m2i[item_id]

    x = mf_feature_matrix(np.array([uidx]), np.array([midx]))
    pred_mf = float(mf_predict(np.array([uidx]), np.array([midx]))[0])
    pred_surr = float(surr.predict(x)[0])

    shap_vals = explainer.shap_values(x)[0]
    base = float(explainer.expected_value)

    title = items.loc[items["item_id"] == item_id, "title"].values
    title = title[0] if len(title) else str(item_id)

    contribs = list(zip(feature_names, shap_vals))
    contribs.sort(key=lambda t: abs(t[1]), reverse=True)

    print(f"\nExplanation for: User {user_id} -> Movie {item_id} ({title})")
    print(f"MF prediction:        {pred_mf:.3f}")
    print(f"Surrogate prediction: {pred_surr:.3f}")
    print(f"SHAP base value:      {base:.3f}")
    print("Top SHAP contributions:")
    for name, val in contribs[:top_n]:
        print(f"  {name:>16}: {val:+.4f}")

    return shap_vals, x

# Example explanation on a real test row
row = test.iloc[0]
_ = explain_prediction(int(row.user_id), int(row.item_id), top_n=10)

/usr/local/lib/python3.13/dist-packages/shap/explainers/_linear.py:123: FutureWarning: The feature_perturbation option is now deprecated in favor of using the appropriate masker (maskers.Independent, maskers.Partition or maskers.Impute).
  warnings.warn(wmsg, FutureWarning)



Surrogate fidelity RMSE (surrogate vs MF): 0.0026900363258669166

Explanation for: User 90 -> Movie 900 (Kundun (1997))
MF prediction:        3.928
Surrogate prediction: 3.929
SHAP base value:      3.523
Top SHAP contributions:
         item_bias: +0.2910
         user_bias: +0.1635
    latent_inter_0: -0.0507
   latent_inter_28: -0.0368
   latent_inter_24: +0.0358
   latent_inter_12: -0.0292
   latent_inter_22: +0.0247
    latent_inter_3: +0.0245
   latent_inter_16: +0.0234
    latent_inter_4: -0.0230


Matrix Factorization predicts a rating using internal parameters (biases and latent vectors), not using explicit  “features” like genre or age.
SHAP, however, explains predictions by attributing them to input features. Because MF does not naturally expose such features, we re-express each MF prediction as a feature vector built from its own components, then train a simple model to reproduce MF’s output. SHAP is applied to this simple model, allowing us to explain which parts of the MF prediction mattered most. The surrogate is a simple, interpretable model that imitates the behaviour of a more complex model so that we can explain its predictions.
The whole idea is that we can EXPLAIN the results we got with MF using a more transparent approach (==explanaible AI)
